In [ ]:
import sys
import os

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally or in Binder")

# Install UppASD and dependencies
print("Installing UppASD from pre-built wheels...")
!pip install -q uppasd --find-links https://uppasd.github.io/UppASD/wheels/
print("Ensuring ase is installed...")
!pip install -q ase
print("✓ All dependencies ready!")

## Cloud Environment Setup

This cell detects if you're running on Google Colab and installs UppASD if needed. It does nothing when running locally or on Binder (where UppASD is pre-installed).

# Multi-Species UppASD Interactive Setup
Create and run atomistic spin dynamics simulations with multiple atom types (Fe, Co, B).

This notebook demonstrates:
1. Define crystal structure with multiple chemical species
2. Set up species-dependent magnetic moments
3. Configure type-resolved exchange interactions
4. Run time evolution dynamics
5. Plot magnetization and energy vs time

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Phase 2 imports for measurement configuration
from uppasd import notebook
from uppasd.configuration_manager import ConfigurationManager, ConfigurationBuilder

%matplotlib inline

## 1. Define Multi-Species Crystal Structure
Create a layered Fe/Co/B system with three atom types.

In [ ]:
# Lattice parameters (simple cubic for demonstration)
alat = 2.5  # Angstrom (lattice scaling factor)

# Lattice vectors (normalized Cartesian unit vectors)
# UppASD uses these as basis, scaled by 'cell' in inpsd.dat
lattice_vectors = np.array([
    [1.0, 0.0, 0.0],  # a1
    [0.0, 1.0, 0.0],  # a2
    [0.0, 0.0, 1.0]   # a3
])

# Basis atoms (fractional coordinates) with atom types
# Format: [x, y, z, type] where type: 1=Fe, 2=Co, 3=B
basis_fractional = np.array([
    [0.0, 0.0, 0.0, 1],  # Fe at corner
    [0.5, 0.5, 0.0, 2],  # Co in-plane center
    [0.5, 0.0, 0.5, 2],  # Co 
    [0.0, 0.5, 0.5, 3],  # B
])

# System size (supercell repetitions)
n1, n2, n3 = 8, 8, 4
natom = len(basis_fractional) * n1 * n2 * n3

print(f"System: {natom} atoms in {n1}×{n2}×{n3} supercell")
print(f"Lattice constant: {alat} Å")
print(f"Basis atoms: {len(basis_fractional)}")
print(f"  Fe atoms: {np.sum(basis_fractional[:, 3] == 1)} per cell")
print(f"  Co atoms: {np.sum(basis_fractional[:, 3] == 2)} per cell")
print(f"  B atoms: {np.sum(basis_fractional[:, 3] == 3)} per cell")

## 2. Generate Atomic Coordinates with Types
Build full supercell preserving atom type information.

In [ ]:
# Generate coordinates using helper function
from uppasd import notebook as nb

coords, atom_types = nb.generate_supercell_coordinates(
    lattice_vectors, basis_fractional, n1, n2, n3, scale=alat
)

print(f"Generated {len(coords)} coordinates")
print(f"Atom type distribution:")
print(f"  Type 1 (Fe): {np.sum(atom_types == 1)} atoms")
print(f"  Type 2 (Co): {np.sum(atom_types == 2)} atoms")
print(f"  Type 3 (B):  {np.sum(atom_types == 3)} atoms")
print(f"\nFirst 5 atoms:")
for i in range(5):
    print(f"  {i + 1}: type={atom_types[i]}, pos={coords[i]}")

## 3. Define Species-Dependent Magnetic Moments
Set different moment magnitudes for Fe, Co, and B.

In [ ]:
# Magnetic moment magnitudes per species (μB)
moments_per_type = {
    1: 2.2,  # Fe
    2: 1.7,  # Co
    3: 0.5   # B (small moment)
}

# Initialize ferromagnetic moments and add perturbation
moments = nb.create_ferromagnetic_moments(natom, moment_magnitude=1.0)

# Apply species-dependent magnitudes and add perturbation
for i, atype in enumerate(atom_types):
    moments[i] *= moments_per_type[atype]

moments = nb.perturb_moments(
    moments, 
    perturbation_scale=0.01, 
    seed=42, 
    renormalize=True,
    moment_magnitudes=np.array([moments_per_type[t] for t in atom_types])
)

print(f"Moments shape: {moments.shape}")
print(f"Mean moment per type:")
for atype in [1, 2, 3]:
    mask = atom_types == atype
    mean_mag = np.mean(np.linalg.norm(moments[mask], axis=1))
    print(f"  Type {atype}: {mean_mag:.3f} μB")

## 4. Define Type-Resolved Exchange Interactions
Set up different J values for Fe-Fe, Fe-Co, Co-Co, Co-B, etc.

In [ ]:
# Exchange parameters (mRy = milli-Rydberg)
# Note: 1 meV ≈ 0.0735 mRy, 1 mRy ≈ 13.6 meV
# 
# Format: {(type_i, type_j, distance_shell): J_value}
# distance_shell: 1=NN (nearest neighbor), 2=NNN (next-nearest), etc.
# 
# This allows different J values for different coordination shells
exchange_matrix = {
    # Nearest neighbor interactions (shell 1)
    (1, 1, 1): 1.544,   # Fe-Fe NN (~21 meV)
    (2, 2, 1): 1.323,   # Co-Co NN (~18 meV)
    (3, 3, 1): 0.368,   # B-B NN (~5 meV)
    (1, 2, 1): 1.433,   # Fe-Co NN (~19.5 meV)
    (2, 1, 1): 1.433,   # Co-Fe NN (symmetric)
    (1, 3, 1): 0.588,   # Fe-B NN (~8 meV)
    (3, 1, 1): 0.588,   # B-Fe NN
    (2, 3, 1): 0.662,   # Co-B NN (~9 meV)
    (3, 2, 1): 0.662,   # B-Co NN
    
    # Next-nearest neighbor interactions (shell 2) - optional, smaller values
    (1, 1, 2): 0.5,     # Fe-Fe NNN
    (2, 2, 2): 0.4,     # Co-Co NNN
    (1, 2, 2): 0.45,    # Fe-Co NNN
    (2, 1, 2): 0.45,    # Co-Fe NNN
}

# Define distance ranges for each shell (in units of alat)
# These define which neighbors belong to which shell
distance_shells = {
    1: (0.0, 0.9),      # NN: 0 < r < 0.9*alat (catches vectors like [0.5,0.5,0])
    2: (0.9, 1.3),      # NNN: 0.9*alat < r < 1.3*alat (catches vectors like [1,0,0])
    3: (1.3, 2.0),      # Further neighbors
}

print("Exchange interaction matrix (mRy):")
print("\nNearest Neighbors (NN):")
print("     Fe    Co    B")
for i in [1, 2, 3]:
    row = []
    for j in [1, 2, 3]:
        val = exchange_matrix.get((i, j, 1), 0.0)
        row.append(f"{val:5.3f}")
    label = ['Fe', 'Co', 'B'][i-1]
    print(f"{label}  " + "  ".join(row))

print("\nNext-Nearest Neighbors (NNN):")
print("     Fe    Co    B")
for i in [1, 2, 3]:
    row = []
    for j in [1, 2, 3]:
        val = exchange_matrix.get((i, j, 2), 0.0)
        row.append(f"{val:5.3f}")
    label = ['Fe', 'Co', 'B'][i-1]
    print(f"{label}  " + "  ".join(row))

## 5. Build Exchange Neighbor List
Use ASE to automatically find neighbors and compute exchange vectors.

The exchange matrix defined above uses a **shell-based** approach:
- Shell 1: Nearest neighbors (NN) - e.g., Fe-Co at (0.5, 0.5, 0.0)
- Shell 2: Next-nearest neighbors (NNN) - e.g., Fe-Fe at (1.0, 0.0, 0.0)
- etc.

The `build_exchange_list` function will:
1. Find all neighbors within cutoff using ASE
2. Calculate distance |r_ij| for each pair
3. Assign to appropriate shell based on distance
4. Look up J_ij value from exchange_matrix
5. Return exchange vectors in fractional coordinates

In [ ]:
# Build exchange list using helper function from uppasd.notebook
exchange_list = nb.build_exchange_list(
    lattice_vectors, 
    basis_fractional[:, :3], 
    basis_fractional[:, 3].astype(int),
    exchange_matrix,
    distance_shells,
    cutoff=4.0,  # Angstroms (large enough to catch NNN)
    scale=alat
)

print(f"✓ Found {len(exchange_list)} unique exchange vectors")
print("  (includes all symmetry-equivalent interactions)")

# Count interactions by type pair and shell
interaction_counts = nb.count_interaction_vectors(
    exchange_list,
    distance_shells,
    type_labels={1: 'Fe', 2: 'Co', 3: 'B'},
    shell_labels={1: 'NN', 2: 'NNN', 3: 'NNNN'}
)

print(f"\nFirst 15 exchange interactions:")
type_labels = {1: 'Fe', 2: 'Co', 3: 'B'}
shell_labels = {1: 'NN', 2: 'NNN', 3: 'NNNN'}
for i, (ti, tj, rx, ry, rz, J) in enumerate(exchange_list[:15]):
    dist = np.sqrt(rx**2 + ry**2 + rz**2)
    shell = '?'
    for s, (d_min, d_max) in distance_shells.items():
        if d_min < dist <= d_max:
            shell = s
            break
    shell_label = {1: 'NN', 2: 'NNN', 3: 'NNNN'}.get(shell, f'S{shell}')
    print(f"  {i+1}. {type_labels[ti]}-{type_labels[tj]} ({shell_label}): "
          f"r=({rx:6.3f}, {ry:6.3f}, {rz:6.3f}), |r|={dist:.3f}, J={J:.3f} mRy")

## 6. Write Input Files
Create UppASD input files using the automatically generated exchange list.

In [ ]:
# Create working directory
work_dir = Path('./multi_species_sim')
work_dir.mkdir(exist_ok=True)

# Create simulation configuration using helper
config = nb.create_relaxation_protocol(
    'mc_sd',
    mc_steps=1000, mc_temp=300,
    sd_steps=5000, sd_temp=100, damping=0.05
)

# Add structure information
config.update({
    'simid': 'notebook',
    'ncell': [n1, n2, n3],
    'cell': np.eye(3),
    'alat': alat,
    'posfile': './posfile.dat',
    'momfile': './momfile.dat',
    'exchange': './jfile.dat',
    'do_prnstruct': 1,
    'Mensemble': 1,
    'Initmag': 3,
    'avrg_step': 10,
    'plotenergy': 1,
})

# Write all input files using helper functions
nb.write_inpsd_file(work_dir / 'inpsd.dat', config)
nb.write_posfile(work_dir / 'posfile.dat', basis_fractional)
nb.write_momfile(work_dir / 'momfile.dat', moments_per_type, 
                 atom_types=basis_fractional[:, 3].astype(int))

print("✓ Created inpsd.dat with MC+SD protocol")
print(f"  alat: {alat} Å, cell: normalized unit vectors")
print(f"✓ Created posfile.dat ({len(basis_fractional)} basis atoms)")
print(f"✓ Created momfile.dat (Fe=2.2, Co=1.7, B=0.5 μB)")

In [ ]:
# Write momfile.dat
# Format: atom_index 1 m_mag m_x m_y m_z
with open(work_dir / 'momfile.dat', 'w') as f:
    for i, atom_data in enumerate(basis_fractional, 1):
        atype = int(atom_data[3])
        mag = moments_per_type[atype]
        f.write(f"{i}  1  {mag:.6f}  0.0 0.0 1.0\n")

print(f"✓ Created momfile.dat")
print(f"  Content preview:")
with open(work_dir / 'momfile.dat', 'r') as f:
    print("  " + f.read())

In [ ]:
# Write exchange interactions using helper
nb.write_jfile(work_dir / 'jfile.dat', exchange_list)

print(f"✓ Created jfile.dat with {len(exchange_list)} interactions")
print(f"\nAll input files ready in: {work_dir}")

## 7. Run Simulation
Execute time evolution and collect trajectory data.

In [ ]:
# Change to working directory
import os
from pathlib import Path
# Ensure work_dir exists (fallback)
if 'work_dir' not in globals():
    work_dir = Path('./multi_species_sim')
    work_dir.mkdir(exist_ok=True)

original_dir = os.getcwd()
try:
    os.chdir(work_dir)
except Exception as e:
    print(f"Could not switch to work_dir {work_dir}: {e}")
    work_dir = Path(original_dir)
    os.chdir(original_dir)

try:
    from uppasd import Simulator
    from uppasd.fileio import UppASDReader

    print("Initializing simulator...")
    with Simulator() as sim:
        print(f"✓ Initialized: {sim.natom} atoms, {sim.mensemble} ensembles")
        
        # Build measurement configuration using Phase 2
        print("\nBuilding measurement configuration...")
        cfg = (ConfigurationBuilder()
            .add_measurement('basic', plotenergy=1, do_avrg='Y', avrg_step=10)
            .build())
        
        # Convert to Fortran kwargs for sim.measure()
        meas_kwargs = cfg.to_fortran_kwargs()
        print(f"  Config keys: {list(meas_kwargs.keys())}")
        print(f"  plotenergy={meas_kwargs.get('plotenergy')}")
        print(f"  do_avrg={meas_kwargs.get('do_avrg')}")
        
        # Run time evolution using relax (integration only)
        print("\nRunning spin dynamics (mode=S)...")
        final_moments = sim.relax(
            mode='S',
            temperature=100,
            steps=5000,
            timestep=1e-16,
            damping=0.05,
        )
        
        # Run measurement phase to write output files
        # CRITICAL: Pass the measurement configuration to enable energy output
        print("\nRunning measurement phase with energy output...")
        files = sim.measure(**meas_kwargs)
        
        # Save values before exiting context manager
        final_energy = sim.energy
        mag_final = np.linalg.norm(np.mean(final_moments, axis=(1, 2)))
        
        print(f"\n✓ Simulation complete")
        print(f"✓ Final energy: {final_energy:.3f}")
        print(f"✓ Final |M|: {mag_final:.3f}")
        print(f"✓ Output files: {files}")
    
    # Load results using UppASDReader (outside the with block)
    try:
        simid = 'notebook'
        reader = UppASDReader(simid=simid)
        avg_data = reader.read_averages()
        ene_data = reader.read_energy()
        
        print(f"\n✓ Loaded results from measurement phase")
        
        # Convert data to numpy arrays, handling potential type issues
        time_data = np.array(ene_data.get('time', []), dtype=float)
        energy_data = np.array(ene_data.get('energy', []), dtype=float)
        mag_data = np.array(avg_data.get('magnetization', []), dtype=float)
        
        if avg_data and 'time' in avg_data:
            print(f"  Averages: {len(avg_data.get('time', []))} points")
        if ene_data and 'time' in ene_data:
            print(f"  Energy: {len(ene_data.get('time', []))} points")
        
        # Extract trajectory data for plotting
        trajectory = {
            'time': time_data,
            'energy': energy_data,
            'magnetization': mag_data,
            'mag_per_type': {1: [], 2: [], 3: []},  # Per-type data if available
        }
    except Exception as e:
        print(f"\nNote: Could not load output files: {e}")
        # Fallback: create minimal trajectory with final state (as numpy arrays)
        trajectory = {
            'time': np.array([5000.0]),
            'energy': np.array([final_energy]),
            'magnetization': np.array([mag_final]),
            'mag_per_type': {1: [], 2: [], 3: []},
        }

except ImportError:
    print("\n⚠️  UppASD Python module not found!")
    print("Install with: pip install -e /path/to/UppASD")
    print("\nAlternatively, run from terminal:")
    print(f"  cd {work_dir}")
    print("  sd < inpsd.dat")
finally:
    os.chdir(original_dir)

## 8. Plot Magnetization vs Time
Visualize total and species-resolved magnetization dynamics.

In [ ]:
from uppasd import plotting
import matplotlib.pyplot as plt

# Build series dict using each measurable's native time array (no interpolation)
timestep_default = (config.get('timestep') if isinstance(globals().get('config', None), dict) else 1e-16)
series = {}
series['total_mag'] = {
    'time': trajectory.get('time', []),
    'values': trajectory.get('magnetization', []),
    'label': 'Total |M|',
    'timestep': timestep_default,
}
# per-type magnetization (if available)
for atype in [1, 2, 3]:
    vals = trajectory.get('mag_per_type', {}).get(atype, [])
    series[f'mag_type_{atype}'] = {
        'time': trajectory.get('time', []),
        'values': vals,
        'label': f'Type {atype}',
        'timestep': timestep_default,
    }
# energy series
series['energy'] = {
    'time': trajectory.get('time', []),
    'values': trajectory.get('energy', []),
    'label': 'Total Energy',
    'timestep': timestep_default,
}

fig, ax = plotting.plot_series(series, to_physical=True, timestep=timestep_default, time_unit='fs', connect=True, show_points=True, title='Magnetization & Energy (native sampling)')
# Save using the returned figure
fig.savefig(work_dir / 'magnetization_vs_time.png', dpi=150, bbox_inches='tight')
print(f"✓ Saved: {work_dir / 'magnetization_vs_time.png'}")
plt.show()

## 9. Plot Energy vs Time
Visualize energy evolution during relaxation.

In [ ]:
from uppasd import plotting
import matplotlib.pyplot as plt

# Plot total energy using native sampling (no interpolation)
timestep_default = (config.get('timestep') if isinstance(globals().get('config', None), dict) else 1e-16)
series = {
    'energy': {
        'time': trajectory.get('time', []),
        'values': trajectory.get('energy', []),
        'label': 'Total Energy',
        'timestep': timestep_default,
    }
}

fig, ax = plotting.plot_series(series, to_physical=True, timestep=timestep_default, time_unit='fs', connect=True, show_points=True, title='Total Energy vs Time')
# Add statistics text if energy array available
ene = trajectory.get('energy', [])
if len(ene) > 0:
    E_initial = float(ene[0])
    E_final = float(ene[-1])
    E_change = E_final - E_initial
    textstr = f'Initial E: {E_initial:.2f}\\nFinal E: {E_final:.2f}\\nΔE: {E_change:.2f}'
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.savefig(work_dir / 'energy_vs_time.png', dpi=150, bbox_inches='tight')
print(f"✓ Saved: {work_dir / 'energy_vs_time.png'}")
plt.show()

## 10. Combined Plot
Show both magnetization and energy on the same figure.

In [ ]:
from uppasd import plotting
import matplotlib.pyplot as plt

timestep_default = (config.get('timestep') if isinstance(globals().get('config', None), dict) else 1e-16)
# Left: magnetization (use native sampling)
series_mag = {
    'total_mag': {'time': trajectory.get('time', []), 'values': trajectory.get('magnetization', []), 'label': 'Total |M|', 'timestep': timestep_default},
}
# Right: energy (native sampling)
series_energy = {
    'energy': {'time': trajectory.get('time', []), 'values': trajectory.get('energy', []), 'label': 'Total Energy', 'timestep': timestep_default},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plotting.plot_series(series_mag, to_physical=True, timestep=timestep_default, time_unit='fs', connect=True, show_points=True, ax=ax1, title='Magnetization Dynamics')
plotting.plot_series(series_energy, to_physical=True, timestep=timestep_default, time_unit='fs', connect=True, show_points=True, ax=ax2, title='Energy Evolution')
plt.suptitle('Multi-Species Fe-Co-B System Dynamics', fontsize=15, fontweight='bold', y=1.02)
fig.savefig(work_dir / 'dynamics_combined.png', dpi=150, bbox_inches='tight')
print(f"✓ Saved: {work_dir / 'dynamics_combined.png'}")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
- ✓ Multi-species system setup (Fe, Co, B)
- ✓ Species-dependent magnetic moments
- ✓ Type-resolved exchange interactions
- ✓ Time evolution dynamics (mode=S)
- ✓ Magnetization vs time analysis
- ✓ Energy vs time tracking

**Key files created:**
- `posfile.dat`: Multi-type atomic positions
- `momfile.dat`: Species-dependent moments
- `jfile.dat`: Type-resolved exchange couplings
- Plots: magnetization and energy dynamics

**Next steps:**
- Modify exchange matrix for different materials
- Add temperature dependence studies
- Analyze correlation functions
- Compute type-resolved susceptibilities